TOOLS

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
model=init_chat_model("google_genai:gemini-3.5-flash-lite")
response=model.invoke("why do parrots talk?")
response

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


AIMessage(content=[{'type': 'text', 'text': 'Parrots are famous for their ability to mimic human speech, but to understand *why* they do it, we have to look at their biology, social behavior, and life in the wild. \n\nHere are the primary reasons why parrots talk:\n\n### 1. Social Integration and Flocking\nIn the wild, parrots are extremely social, flock-dwelling birds. Survival depends on fitting in and communicating with the group. To strengthen bonds, parrots constantly mimic the calls of their mates and flock members. When a parrot lives in a human home, **humans become its "flock."** Mimicking human words is their way of trying to connect, bond, and fit in with their human family.\n\n### 2. Desire for Attention and Interaction\nParrots are highly intelligent and emotional animals that require a lot of mental stimulation. They quickly learn that making certain sounds—like "Hello!", "Want a cracker?", or even the microwave oven beep—gets an immediate reaction from humans. Whether yo

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It is sunny in {location}"

model_with_tools=model.bind_tools([get_weather])

In [10]:
response=model_with_tools.invoke("Whats the weather like in boston?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'call_1558803': 'El4KXAERTTIPMasedOnCiTiHLP7vB6e+yEf0oE85lnIKPYzIbdf8POFOhHylXKPK/vYf8AYyHkTZ3CUHd+wBOSrKvafjMIYVgJB8SIU+AxwZQL5IOyTnSLfTSrO2ZILr'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a06234-c044-7f23-9de0-d6cb5fbaa891-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_1558803', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 51, 'output_tokens': 16, 'total_tokens': 67, 'input_token_details': {'cache_read': 0}}
Tool: get_weather
Args: {'location': 'Boston'}


In [3]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny.


In [4]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'call_1534929': 'El4KXAERTTIPE5MFBLyz2kpv9lUbwrr+fLp6j3qMEKEdmQp1SBNNv4SJFfZ3xR7XNdjctMXLgbwe0MU1aYh2HXzr7N6gw3pK+rdUpnqVfvw5Owof+YjHkzlqbpen0qpB'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0668c-0dd6-7ca0-9d8f-d4b1e94ea3ac-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_1534929', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 16, 'total_tokens': 68, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='It is sunny in Boston', name='get_weather', tool_call_id='call_1534929')]